### Task 1.

Сколько несовместных независимых экспериментов можно запустить одновременно, если за время эксперимента собираем 10000 наблюдений?

Если решаем запустить 10 экспериментов, то на каждый эксперимент можно выделить по 1000 наблюдений, размер групп будет равен 500.

Параметры экспериментов:

- проверяем гипотезу о равенстве средних;
- уровень значимости — 0.05;
- допустимая вероятность ошибки II рода — 0.1;
- ожидаемый эффект — увеличение значений на 3%;
- способ добавления эффекта в синтетических А/Б экспериментах — умножение на константу.

Будем считать, что распределение измеряемых величин является нормальным распределением со средним 100 и стандартным отклонением 10.

В качестве ответа введите максимально возможное количество экспериментов, которое можно запустить с указанными выше параметрами.

**Решение**

In [1]:
import numpy as np
from scipy import stats

In [2]:
# параметры эксперимента
count_all = 10000
sample_mean = 100
sample_std = 10
effect = 0.03
alpha = 0.05
beta = 0.1

# размер групп
def estimate_sample_size(effect, std, alpha, beta):
    t_alpha = stats.norm.ppf(1 - alpha / 2, loc=0, scale=1)
    t_beta = stats.norm.ppf(1 - beta, loc=0, scale=1)
    var = 2 * std ** 2
    sample_size = int((t_alpha + t_beta) ** 2 * var / (effect ** 2))
    return sample_size

In [3]:
# необходимый размер групп
sample_size = estimate_sample_size(effect * sample_mean, sample_std, alpha, beta)
# количество экспериментов
count_exp = count_all / (sample_size * 2)
print(f'count_exp = {count_exp:0.1f}')

count_exp = 21.5


**Ответ**: 21

### Task 2.

Задача похожа на предыдущую, только теперь решение принимается не независимо для каждого эксперимента.
Например, у нас есть 5 текстов для маркетинговой рассылки, хотим проверить, какой эффективнее работает и работает ли вообще.
Алгоритм будет следующий:

1. Формируем непересекающиеся контрольные и экспериментальные группы для каждого из 5 вариантов.
2. Проводим параллельно 5 экспериментов.
3. С помощью метода Холма определяем, в каких экспериментах были статистически значимые отличия.
4. Если значимых отличий не обнаружено, то говорим, что эффекта нет, все варианты отклоняем.
5. Если значимые отличия обнаружены, то из вариантов со значимым эффектом выбираем вариант с наименьшим значением p-value, будем использовать его.

Будем считать, что совершена ошибка I рода, если найдены значимые отличия, когда на самом деле их не было ни в одном из вариантов.

Будем считать, что совершена ошибка II рода, если:
- либо не найдено значимых отличий, когда на самом деле они были;
- либо выбранный для дальнейшего использования вариант на самом деле был без эффекта, при этом были варианты с эффектом.

Параметры экспериментов:

- проверяем гипотезу о равенстве средних;
- уровень значимости — 0.05;
- допустимая вероятность ошибки II рода — 0.1;
- ожидаемый эффект — увеличение значений на 3%;
- способ добавления эффекта в синтетических А/Б экспериментах — умножение на константу.

Замечание: при оценке вероятности ошибки II рода нужно рассматривать худший сценарий, когда эффект есть только в одном из экспериментов. Чем в большем количестве экспериментов будет эффект, тем меньше будет вероятность ошибки II рода.

Будем считать, что распределение измеряемых величин является нормальным распределением со средним 100 и стандартным отклонением 10.

В качестве ответа введите максимально возможное количество экспериментов, которое можно запустить с указанными выше параметрами.

**Решение**

In [4]:
# параметры эксперимента
count_all = 10000
sample_mean = 100
sample_std = 10
effect = 0.03
alpha = 0.05
beta = 0.1

In [5]:
def method_holm(pvalues, alpha=0.05):
    """Применяет метод Холма для проверки значимости изменений.

    pvalues - List[float] - список pvalue.
    alpha - float, уровень значимости.
    return - np.array, массив из нулей и единиц, 0 - эффекта нет, 1 - эффект есть.
    """
    m = len(pvalues)
    array_alpha = np.arange(m, 0, -1)
    array_alpha = alpha / array_alpha
    sorted_pvalue_indexes = np.argsort(pvalues)
    res = np.zeros(m)
    for idx, pvalue_index in enumerate(sorted_pvalue_indexes):
        pvalue = pvalues[pvalue_index]
        alpha_ = array_alpha[idx]
        if pvalue < alpha_:
            res[pvalue_index] = 1
        else:
            break
    res = res.astype(int)
    return res

In [6]:
def estimate_ci_bernoulli(p, n, alpha=0.05):
    """Доверительный интервал для Бернуллиевской случайной величины."""
    t = stats.norm.ppf(1 - alpha / 2, loc=0, scale=1)
    std_n = np.sqrt(p * (1 - p) / n)
    return p - t * std_n, p + t * std_n

In [7]:
for count_exp in range(11, 15):
    errors_aa = []
    errors_ab = []
    sample_size = int(count_all / (int(count_exp) * 2))
    for _ in range(5000):
        list_ab_values = [
            np.random.normal(sample_mean, sample_std, (2, sample_size))
            for _ in range(count_exp)
        ]
        # синтетический А/А тест
        pvalues = [stats.ttest_ind(a, b).pvalue for a, b in list_ab_values]
        aa_with_effect = method_holm(pvalues, alpha)
        errors_aa.append(np.sum(aa_with_effect) > 0)

        # Синтетический А/Б тест.
        # Добавим эффект в первый эксперимент (не важно в какой добавлять, так как данные случайные)
        list_ab_values[0][1] *= 1 + effect
        pvalues = [stats.ttest_ind(a, b).pvalue for a, b in list_ab_values]
        ab_with_effect = method_holm(pvalues, alpha)
        if np.sum(ab_with_effect) == 0:
            # если эффектов не найдено, то это ошибка
            errors_ab.append(True)
        else:
            # если эффектов найден где его нет, то это ошибка
            errors_ab.append(np.min(pvalues) != pvalues[0])

    estimated_first_type_error = np.mean(errors_aa)
    estimated_second_type_error = np.mean(errors_ab)
    ci_first = estimate_ci_bernoulli(estimated_first_type_error, len(errors_aa))
    ci_second = estimate_ci_bernoulli(estimated_second_type_error, len(errors_ab))
    print(f'count_exp = {count_exp}')
    print(f'sample_size = {sample_size}')
    print(f'оценка вероятности ошибки I рода = {estimated_first_type_error:0.4f}')
    print(f'  доверительный интервал = [{ci_first[0]:0.4f}, {ci_first[1]:0.4f}]')
    print(f'оценка вероятности ошибки II рода = {estimated_second_type_error:0.4f}')
    print(f'  доверительный интервал = [{ci_second[0]:0.4f}, {ci_second[1]:0.4f}]')

count_exp = 11
sample_size = 454
оценка вероятности ошибки I рода = 0.0446
  доверительный интервал = [0.0389, 0.0503]
оценка вероятности ошибки II рода = 0.0556
  доверительный интервал = [0.0492, 0.0620]
count_exp = 12
sample_size = 416
оценка вероятности ошибки I рода = 0.0502
  доверительный интервал = [0.0441, 0.0563]
оценка вероятности ошибки II рода = 0.0802
  доверительный интервал = [0.0727, 0.0877]
count_exp = 13
sample_size = 384
оценка вероятности ошибки I рода = 0.0476
  доверительный интервал = [0.0417, 0.0535]
оценка вероятности ошибки II рода = 0.1194
  доверительный интервал = [0.1104, 0.1284]
count_exp = 14
sample_size = 357
оценка вероятности ошибки I рода = 0.0448
  доверительный интервал = [0.0391, 0.0505]
оценка вероятности ошибки II рода = 0.1640
  доверительный интервал = [0.1537, 0.1743]


**Ответ**: 12